Biometrich Technologies and Behavioural Security
# **<center>Tutorial 7 - Behavioral biometric: EEG signal </center>**
### <center> Part 3 </center>

##Step 0: Reload data

In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pandas import read_csv
from numpy import mean
from numpy import std
from numpy import delete
from numpy import savetxt
from matplotlib import pyplot

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

downloaded = drive.CreateFile({'id': '1R9SnVSeJoSCwyNldRlxMneB6_g3DoD6Y'})
downloaded.GetContentFile('EEG Eye State.txt')        # replace the file name with your file

# load the dataset
data = read_csv('/content/EEG Eye State.txt', header=None)
# retrieve data as numpy array
values = data.values
for i in range(values.shape[1] - 1):
	# calculate column mean and standard deviation
	data_mean, data_std = mean(values[:,i]), std(values[:,i])
	# define outlier bounds
	cut_off = data_std * 4
	lower, upper = data_mean - cut_off, data_mean + cut_off
	# remove too small
	too_small = [j for j in range(values.shape[0]) if values[j,i] < lower]
	values = delete(values, too_small, 0)
	print('>deleted %d rows' % len(too_small))
	# remove too large
	too_large = [j for j in range(values.shape[0]) if values[j,i] > upper]
	values = delete(values, too_large, 0)
	print('>deleted %d rows' % len(too_large))
# save the results to a new file
savetxt('EEG_Eye_State_no_outliers.csv', values, delimiter=',')


>deleted 0 rows
>deleted 1 rows
>deleted 2 rows
>deleted 1 rows
>deleted 0 rows
>deleted 142 rows
>deleted 0 rows
>deleted 48 rows
>deleted 0 rows
>deleted 153 rows
>deleted 0 rows
>deleted 43 rows
>deleted 0 rows
>deleted 0 rows
>deleted 0 rows
>deleted 15 rows
>deleted 0 rows
>deleted 5 rows
>deleted 10 rows
>deleted 0 rows
>deleted 21 rows
>deleted 53 rows
>deleted 0 rows
>deleted 12 rows
>deleted 58 rows
>deleted 53 rows
>deleted 0 rows
>deleted 59 rows


### Step 2.3 Repeat split and train with no shuffle

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
X, y = values[:, :-1], values[:, -1]
# split the dataset
trainX, testX, trainy, testy = train_test_split(X, y, test_size=0.1, shuffle=False, random_state=1)

model = KNeighborsClassifier(n_neighbors=3)
# fit model on train set
model.fit(trainX, trainy)
# forecast test set
predictions = model.predict(testX)
# evaluate predictions
acc='Total Accuracy: %.2f %%' % (accuracy_score(testy, predictions)*100)
print(acc)

Total Accuracy: 52.69 %


Pretty low accuracy. Maybe the last 10% of dataset is hard to predict. We can repeat the experiment and use the first 10% of the data in time for test and the last 90% for train.

In [ ]:
from numpy import flip
fvalues = flip(values, 0)

X, y = fvalues[:, :-1], fvalues[:, -1]
# split the dataset
trainX, testX, trainy, testy = train_test_split(X, y, test_size=0.1, shuffle=False, random_state=1)

model = KNeighborsClassifier(n_neighbors=3)
# fit model on train set
model.fit(trainX, trainy)
# forecast test set
predictions = model.predict(testX)
# evaluate predictions
acc='Total Accuracy: %.2f %%' % (accuracy_score(testy, predictions)*100)
print(acc)

Total Accuracy: 52.90 %


A similar value is achieved by the NN. You can check if you want. But let's move on.

## Step 3: Finding solutions

### Step 3.1 Other classifiers

Try with a different classifier. For example SVM. The accuracy is better or worse? Why?

In [ ]:
X, y = values[:, :-1], values[:, -1]
# split the dataset
trainX, testX, trainy, testy = train_test_split(X, y, test_size=0.1, shuffle=False, random_state=1)

from sklearn.svm import SVC

model = SVC()
model.fit(trainX, trainy)

predictions = model.predict(testX)

acc='SVM: Total Accuracy: %.2f %%' % (accuracy_score(testy, predictions)*100)
print(acc)

SVM: Total Accuracy: 80.92 %


### Step 3.2 Statistical Features approach

Another approach consists of extracting statistical feature from the signal and use them to train the classifier.

#### Feature Extraction

In [ ]:
import pandas as pd
import numpy as np
import scipy
from scipy import stats

X_columns = ['mean', 'standard deviation', 'kurt', 'skewness']
Y_columns = ['label']
x, y = values[:, :-1], np.array(values[:, -1], dtype= 'int32')
X = pd.DataFrame(columns = X_columns)
Y = pd.DataFrame(columns = Y_columns)
for i in range(len(x)):
  X.loc[i] = np.array([np.mean(x[i]), np.std(x[i]), stats.kurtosis(x[i]), stats.skew(x[i])])
  Y.loc[i] = y[i]

In [ ]:
from sklearn.model_selection import train_test_split

# Split the 'features' and 'income' data into training and testing sets
X_train1, X_test1, y_train1, y_test1 = train_test_split(X,
                                                    y,
                                                    shuffle='False',
                                                    test_size = 0.2,
                                                    random_state = 0)



After a couple of tries, I found that Random forest are the most suitable classifier for this tasks. But you can explore others, this is only an example.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train1, y_train1)
# forecast test set
predictions = model.predict(X_test1)
# evaluate predictions
acc='RF: Total Accuracy: %.2f %%' % (accuracy_score(y_test1, predictions)*100)
print(acc)

RF: Total Accuracy: 72.56 %


## Conclusions

Let’s review what we have learned today:

*   The model evaluation methodology must take the temporal ordering of observations into account.

This means that it is methodologically invalid to use a train/test split that shuffles the data prior to splitting.

We saw this in the evaluation of the high skill of the model with KNN and NN whith shuffled train/test split compared to the low skill of the model when directly adjacent observations in time were not available at prediction time.

* The model evaluation methodology must make sense for the use of the final model.

Keep in mind that even if you use a methodology that respects the temporal ordering of the observations, the model should only have information available that it would have if the model were being used in practice.